# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset Title: ", metadata.name)
print("Description: ", metadata.description)

# Optionally: show more metadata fields
print("Identifier: ", getattr(metadata, 'identifier', None))
print("Version: ", getattr(metadata, 'version', None))
print("License: ", getattr(metadata, 'license', None))

## 2. Data Overview
Review available record sets, fields, and their IDs.

All entities in this dataset (record sets, fields/columns) are referenced by their `@id` according to the [Croissant schema](https://mlcommons.org/croissant/).

In [ ]:
# List record sets by @id
print("Available Record Sets and their fields:\n")
for record_set in metadata.record_sets:
    print(f"- Record Set name: {record_set.name} | @id: {record_set.id}")
    fields_string = ', '.join(f"{f.name} (@id: {f.id})" for f in record_set.fields)
    print(f"  Fields: {fields_string if fields_string else 'None found'}\n")

# Save a list of all record set @ids for the next steps
record_set_ids = [rs.id for rs in metadata.record_sets]
record_set_ids

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from all record sets into pandas DataFrames
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for record_set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    else:
        print(f"No records found for {rs_id}")

# Choose the first available record set with data for EDA
chosen_record_set_id = None
for k, v in dataframes.items():
    if not v.empty:
        chosen_record_set_id = k
        break
if chosen_record_set_id is None:
    raise ValueError('No non-empty record set found!')

print(f"\nChosen record set for further analysis: {chosen_record_set_id}")
print(dataframes[chosen_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming distributions, or grouping data.

In [ ]:
# Inspect numeric fields for the chosen record set
df = dataframes[chosen_record_set_id]
print("Columns in chosen record set:", df.columns.tolist())

# Find candidate numeric columns (float or int)
sample = df.head(50)
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(sample[col])]
print("Numeric fields identified:", numeric_fields)

if not numeric_fields:
    raise ValueError('No numeric fields found for EDA!')

# Pick the first numeric field for demonstration
numeric_field = numeric_fields[0]
print(f"Using field '{numeric_field}' for numeric analysis.")

threshold = sample[numeric_field].mean()  # use average as an example threshold
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f} (n={len(filtered_df)}):")
display(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to select a grouping field (preferably string/categorical)
group_fields = [col for col in df.columns if df[col].dtype == object and df[col].nunique() < 20]
if group_fields:
    group_field = group_fields[0]
    print(f"\nGrouping by '{group_field}':")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    display(grouped_df.head())
else:
    print("No suitable categorical/grouping field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot for numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=25, color='skyblue')
plt.title(f"Distribution of '{numeric_field}' in record set '{chosen_record_set_id}'")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If a grouping field was found, plot group means
if 'group_field' in locals():
    plt.figure(figsize=(10, 4))
    order = grouped_df.sort_values(numeric_field, ascending=False)[group_field]
    sns.barplot(x=group_field, y=numeric_field, data=grouped_df, order=order)
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.ylabel(f"Mean {numeric_field}")
    plt.xlabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we've loaded and explored the **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** dataset using the `mlcroissant` library by referencing all entities via their `@id`s.

Key steps:
- Loaded dataset schema and explored metadata and structure.
- Examined available record sets and fields by `@id` for precise data extraction and manipulation.
- Loaded data into pandas DataFrames for EDA, selected and visualized numeric fields, and optionally grouped data to reveal patterns.

For more advanced analyses, repeat similar steps using other record sets and field `@id`s. Check the [Croissant documentation](https://mlcommons.org/croissant/) and the dataset [FAIR² schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) for further details.